In [1]:
import ee
import geemap
import numpy as np

ee.Initialize(project='wind-field-estimation-497821')


In [2]:
aoi = ee.Geometry.Rectangle([68.0, 21.0, 70.0, 23.0])

In [3]:
era5 = (
    ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY')
    .filterBounds(aoi)
    .filterDate('2024-06-01', '2024-06-02')
    .select([
        'u_component_of_wind_10m',
        'v_component_of_wind_10m'
    ])
)

image = era5.first()

print(image.bandNames().getInfo())

['u_component_of_wind_10m', 'v_component_of_wind_10m']


In [4]:
Map = geemap.Map()

Map.addLayer(
    image.select('u_component_of_wind_10m'),
    {'min': -10, 'max': 10},
    'U Wind'
)

Map.addLayer(
    image.select('v_component_of_wind_10m'),
    {'min': -10, 'max': 10},
    'V Wind'
)

Map.centerObject(aoi, 8)

Map

Map(center=[22.000678814980056, 69.00000000000001], controls=(WidgetControl(options=['position', 'transparent_…

In [5]:
u = image.select('u_component_of_wind_10m')
v = image.select('v_component_of_wind_10m')

wind_speed = u.pow(2).add(v.pow(2)).sqrt()

Map = geemap.Map()

Map.addLayer(
    wind_speed,
    {'min': 0, 'max': 15},
    'Wind Speed'
)

Map.centerObject(aoi, 8)

Map

Map(center=[22.000678814980056, 69.00000000000001], controls=(WidgetControl(options=['position', 'transparent_…

In [6]:
wind_direction = v.atan2(u)

Map = geemap.Map()

Map.addLayer(
    wind_direction,
    {'min': -3.14, 'max': 3.14},
    'Wind Direction'
)

Map.centerObject(aoi, 8)

Map

Map(center=[22.000678814980056, 69.00000000000001], controls=(WidgetControl(options=['position', 'transparent_…

In [7]:
#getting wind values numerically
wind_speed_info = wind_speed.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi,
    scale=1000
)

print(wind_speed_info.getInfo())

{'u_component_of_wind_10m': 4.846191315944521}
